In [3]:
import os

os.chdir("../")

In [11]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test: Path
    model_path: Path
    params: dict
    metric_file: Path
    target_column: str
    mlflow_uri: str

In [12]:
import ares.constants as const
from ares.utils.common import read_yaml, create_directories, save_json

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=const.CONFIG_FILE_PATH,
        params_filepath=const.PARAMS_FILE_PATH,
        schema_filepath=const.SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.CatBoost
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        mlflow_uri = os.environ.get(
            "MLFLOW_TRACKING_URI", "sqlite:///experiments/mlflow.db"
        )

        model_trainer_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test=config.test,
            model_path=config.model_path,
            target_column=schema.name,
            params=params,
            mlflow_uri=mlflow_uri,
            metric_file=config.metric_file,
        )

        return model_trainer_config

In [14]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.models.signature import infer_signature
from pathlib import Path
import mlflow
import mlflow.catboost

In [ ]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_experiment("Model_Evaluation_Experiment")

        with mlflow.start_run():
            preds = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, preds)

            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file), data=scores)

            mlflow.log_params(self.config.params)
            mlflow.log_metrics({"rmse": rmse, "r2": r2, "mae": mae})

            signature = infer_signature(test_x, preds)

            mlflow.catboost.log_model(
                cb_model=model,
                artifact_path="model",
                signature=signature,
                registered_model_name="CatBoostRegressor",
            )

In [18]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluator = ModelEvaluation(config=model_evaluation_config)
    model_evaluator.log_into_mlflow()
except Exception as e:
    raise e

[2026-02-01 15:43:58,093: INFO: common: YAML file: config/config.yaml loaded successfully]
[2026-02-01 15:43:58,099: INFO: common: YAML file: params.yaml loaded successfully]
[2026-02-01 15:43:58,104: INFO: common: YAML file: schema.yaml loaded successfully]
[2026-02-01 15:43:58,106: INFO: common: Created directory at: artifacts]
[2026-02-01 15:43:58,107: INFO: common: Created directory at: artifacts/model_evaluation]
[2026-02-01 15:43:58,164: INFO: common: JSON file saved at: artifacts/model_evaluation/metrics.json]


/home/cecil/projects/ares/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/02/01 15:43:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/01 15:44:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to 